# Lecture 6.3 — The `trace()` context manager for custom workflow names

**Section 06 — Tracing, Observability & Capstone**

Every call to `Runner.run()` you have written so far created its own separate trace. That is fine when a run
is the whole story. It stops being fine the moment a single logical workflow is made of several runs.

In this notebook you will:

| Step | What you do |
|---|---|
| 1 | Watch two runs produce two unrelated trace IDs |
| 2 | Wrap the same two runs in `with trace(...)` and collapse them into one trace |
| 3 | Prove the nesting is real with `get_current_trace()` |
| 4 | Pass full parameters to `trace()` and get one deep link for the whole workflow |
| 5 | Wrap a concurrent `asyncio.gather()` fan-out in a single trace |
| 6 | Time non-agent work with `custom_span()` |
| 7 | Meet the silent `NoOpSpan` failure that costs people hours |
| 8 | Force an immediate export with `flush_traces()` |

## Cell 1: Install the OpenAI Agents SDK

This cell installs the `openai-agents` package into the current Colab runtime. Everything in this notebook
comes from that one package: the `Agent` and `Runner` classes, and the tracing helpers `trace`,
`custom_span`, `get_current_trace` and `flush_traces`.

The version is pinned so the output you see matches the output recorded for this lecture. If the package is
already present in this runtime, pip confirms it and moves on quickly. The `-q` flag keeps the install log short.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00


## Cell 2: Configure your OpenAI API key

The SDK reads your API key from the `OPENAI_API_KEY` environment variable. Tracing uses the same key: trace
data is exported to the OpenAI platform under the account that key belongs to, which is how the runs in this
notebook end up visible in the Traces dashboard.

**To add the key as a Colab secret:**

1. Click the key icon in the left sidebar of Colab to open the **Secrets** panel.
2. Click **Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY` exactly, with no spaces.
4. Paste your key into the **Value** field.
5. Turn on the **Notebook access** toggle for this notebook.

`userdata.get()` then reads the secret and writes it into `os.environ` for the rest of the session.

**Running locally instead of Colab?** Skip this cell and set the variable in your terminal before launching
Jupyter: `export OPENAI_API_KEY="your-key-here"` on macOS or Linux, or
`setx OPENAI_API_KEY "your-key-here"` on Windows.

In [2]:
import os

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## Cell 3: Declare the model name

`MODEL_NAME` is declared once here and reused in every `Agent` definition below. Change the string in this
one cell and every agent in the notebook switches model. Nothing else needs editing.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

print("Using model:", MODEL_NAME)

Using model: gpt-5.4-mini


## Cell 4: Imports

All imports for the notebook live in this one cell. Four of these names are new in this lecture.

| Import | What it does | Status |
|---|---|---|
| `trace` | Opens a trace context. Runs inside it join that trace instead of creating their own. | **New** |
| `custom_span` | Creates a span for work that is not an agent run, so it appears in the trace tree. | **New** |
| `get_current_trace` | Returns the currently active `Trace` object, or `None` if there is no active trace. | **New** |
| `flush_traces` | Blocks until buffered traces and spans have been exported. | **New** |
| `gen_trace_id` | Generates a correctly formatted trace ID. Introduced in Lecture 6.2. First used in this notebook in Cell 8. | Returning |
| `RunConfig` | Per-run configuration, including `workflow_name` and `trace_id`. | Returning |
| `Agent`, `Runner` | The core run loop. | Returning |
| `ModelSettings`, `Reasoning` | Keep responses short and fast for these demos. | Returning |
| `asyncio`, `time` | Concurrency for the fan-out demo, and simple timing. | Returning |

Note the import path for `Reasoning`. It comes from `openai.types.shared`, not from `agents`. That one
catches people out regularly.

In [4]:
import asyncio
import time

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ModelSettings,
    RunConfig,
    Runner,
    custom_span,
    flush_traces,
    gen_trace_id,
    get_current_trace,
    trace,
)

print("Imports ready.")

Imports ready.


---

## The problem: one run, one trace

Here is something that has been true of every notebook you have written in this course, and that nobody
pointed out until now.

**Every call to `Runner.run()` creates its own trace.**

That is invisible while a workflow is a single run. It becomes very visible the moment a workflow is several
runs:

| Lecture | What you built | Traces produced |
|---|---|---|
| 5.6 | Deterministic pipeline: outline, then draft, then polish | **Three** separate traces |
| 5.7 | Parallel fan-out with `asyncio.gather()` | **Four** separate traces |

In both cases the runs formed one logical workflow in your head, and three or four unrelated entries in the
dashboard. `RunConfig(group_id=...)` helped: it stamped a shared label on each trace so you could filter them
together. But a shared label is not a shared trace. Each run still had its own trace ID, its own root, and its
own timeline. You could not see the whole workflow in one view.

The goal for the rest of this notebook is simple: **make multiple runs part of one trace.**

## Cell 5: Two runs, two separate traces

Before fixing the problem, see it clearly, in the dashboard rather than in printed IDs.

This cell defines a joke-generating agent that you will reuse for the rest of the notebook.
`ModelSettings(reasoning=Reasoning(effort="none"), verbosity="low")` keeps the responses short and the runs
fast, which matters when you are about to make a lot of them.

Then it makes two calls to `Runner.run()`, and both calls pass the exact same `RunConfig(workflow_name=...)`.
That is deliberate. If the two runs had different names, you could talk yourself into believing the names
were somehow what kept them apart. Giving them the identical name removes that excuse: whatever separates
these two traces, it is not the label.

Neither `RunConfig` specifies a `trace_id`. You do not need to generate one by hand: when a trace is created
with no `trace_id`, the SDK generates one for you automatically. That is one less thing to manage, and it is
the normal way to work unless you specifically need to know the ID in advance (you will do exactly that in
Cell 8).

The second run depends on the first: it rates the joke the first run produced. Logically that is one
workflow. After running this cell, open your Traces dashboard and search for **"Separate traces demo"**.
You will find two entries with that same name, each with its own trace ID and its own timeline.

In [5]:
agent = Agent(
    name="Joke generator",
    instructions="Tell funny jokes. Keep them short.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

first = await Runner.run(
    agent,
    "Tell me a joke about databases.",
    run_config=RunConfig(workflow_name="Separate traces demo"),
)

second = await Runner.run(
    agent,
    f"Rate this joke out of 10: {first.final_output}",
    run_config=RunConfig(workflow_name="Separate traces demo"),
)

print("Joke:", first.final_output)
print("Rating:", second.final_output)

print("\nOpen your dashboard and search for 'Separate traces demo'.")
print("You will find TWO separate trace entries with that same name.")

Joke: Why did the database break up with the spreadsheet?

It couldn’t handle the relationship without proper **relations**.
Rating: 7/10 — nerdy, clean, and the “relations” pun lands nicely.

Open your dashboard and search for 'Separate traces demo'.
You will find TWO separate trace entries with that same name.


## Cell 6: The fix, one trace for both runs

Same two runs, same prompts as Cell 5. One structural change, and a new workflow name so the dashboard makes
the contrast obvious against the "Separate traces demo" entries you just found.

This cell wraps both `Runner.run()` calls inside `with trace("Single trace demo"):`. No `RunConfig` this
time, on either call. Where Cell 5 gave two runs the same name and still produced two traces, this cell gives
neither run a name of its own at all, and produces exactly one.

The SDK documentation states the effect directly: because the two calls to `Runner.run` are wrapped in a
`with trace()`, the individual runs become part of the overall trace rather than creating two traces.

### Why it works

The mechanic is one conditional. Verified from `src/agents/tracing/context.py`, in `create_trace_for_run`:

```python
current_trace = get_current_trace()
if current_trace:
    return None
```

When a run starts, the SDK asks whether a trace is already active. If one is, it returns `None` instead of a
new trace, and the run's spans attach to the trace that already exists. That single check is the entire
feature.

### About the signature

`workflow_name` is the first positional argument to `trace()`, which is why `trace("Single trace demo")`
reads the way it does. The full verified signature is:

```python
trace(
    workflow_name: str,
    trace_id: str | None = None,
    group_id: str | None = None,
    metadata: dict[str, Any] | None = None,
    tracing: TracingConfig | None = None,
    disabled: bool = False,
) -> Trace
```

In [6]:
with trace("Single trace demo"):
    first_result = await Runner.run(
        agent,
        "Tell me a joke about databases.",
    )
    second_result = await Runner.run(
        agent,
        f"Rate this joke out of 10: {first_result.final_output}",
    )
    print(f"Joke: {first_result.final_output}")
    print(f"Rating: {second_result.final_output}")

print("\nOpen your dashboard and search for 'Single trace demo'.")
print("You will find ONE trace entry containing both runs.")

Joke: Why did the database break up with the spreadsheet?

It found the spreadsheet too **table**-drama.
Rating: 7/10 — solid pun. Very row-mantic.

Open your dashboard and search for 'Single trace demo'.
You will find ONE trace entry containing both runs.


## Cell 7: Proving the nesting with `get_current_trace()`

Cell 6 already proved the nesting once, in the dashboard. This cell proves the same thing again without
leaving the notebook.

`get_current_trace()` returns the currently active `Trace` object, or `None` when there is no active trace.
The current trace is tracked in a Python `contextvar`, short for context variable, from the standard
library's `contextvars` module. Think of it as a value that belongs to the current execution path rather
than to any one function: code running anywhere inside the `with trace(...)` block can read it, no matter how
deeply nested, without it ever being passed down as an argument. That is what makes it visible to code deep
inside `Runner.run()`, even though `trace()` never hands the trace object to `Runner.run()` directly. Keep
this in mind for Cell 9, where the same mechanism is what lets concurrent runs find the right trace with no
extra effort from you.

This cell calls `get_current_trace()` three times: once before the block, once inside it, and once after it
closes. Inside the block it also runs the agent and prints the outer trace's ID alongside it.

The `Trace` object exposes two properties worth knowing:

| Property | Returns |
|---|---|
| `.trace_id` | The trace ID, formatted `trace_<32 hex chars>` |
| `.name` | The workflow name you passed as the first argument |

Watch the three printed results in order. The transition from `None` to a real object and back again is the
`contextvar` doing its job.

In [7]:
print("Outside any trace:")
print(f"  get_current_trace() -> {get_current_trace()}")

with trace("Nesting demo") as t:
    current = get_current_trace()
    print("\nInside the trace block:")
    print(f"  get_current_trace() -> {type(current).__name__}")
    print(f"  trace_id: {current.trace_id}")
    print(f"  name: {current.name}")

    result = await Runner.run(agent, "Tell me a short joke.")
    print(f"  Run completed inside trace: {current.trace_id}")

print("\nAfter the trace block:")
print(f"  get_current_trace() -> {get_current_trace()}")

Outside any trace:
  get_current_trace() -> None

Inside the trace block:
  get_current_trace() -> TraceImpl
  trace_id: trace_4579d0f5f51a429da00f3123217980ca
  name: Nesting demo
  Run completed inside trace: trace_4579d0f5f51a429da00f3123217980ca

After the trace block:
  get_current_trace() -> None


## Cell 8: Full parameters and one deep link per workflow

In Lecture 6.2 you built a deep link by generating a trace ID and passing it to `RunConfig(trace_id=...)`.
That gave you one link per run. `trace()` takes the same parameters, so now you get one link for an entire
workflow.

This cell generates a trace ID, prints the dashboard URL built from it, then opens a `trace()` block using
that ID plus two more parameters:

| Parameter | Purpose |
|---|---|
| `trace_id` | Your own ID, so you can build the dashboard URL before the workflow runs |
| `group_id` | Links several traces from the same conversation or process, exactly as it did on `RunConfig` |
| `metadata` | An arbitrary dictionary attached to the trace, useful for filtering later |

Both runs happen inside the block, so both land under that one trace ID and that one URL.

This is the pattern the SDK's own `research_bot` example uses: generate the ID, print the link, then open the
trace. Lecture 6.2 deliberately used `RunConfig` instead so that `trace()` could land properly here.

**One caution.** Calling `trace()` while another trace is already active logs a warning that a trace already
exists and this is probably a mistake. The SDK will create the nested trace, but it is almost always not what
you meant. Open one trace per workflow.

In [8]:
workflow_trace_id = gen_trace_id()

print(
    f"View this workflow: "
    f"https://platform.openai.com/traces/trace"
    f"?trace_id={workflow_trace_id}"
)

with trace(
    "Joke pipeline",
    trace_id=workflow_trace_id,
    group_id="joke-session-001",
    metadata={"lecture": "6.3", "demo": "full-params"},
):
    joke = await Runner.run(agent, "Tell me a joke about caching.")
    rating = await Runner.run(
        agent,
        f"Rate this joke out of 10: {joke.final_output}",
    )
    print(f"\nJoke: {joke.final_output}")
    print(f"Rating: {rating.final_output}")

View this workflow: https://platform.openai.com/traces/trace?trace_id=trace_9e80aa88942f4dd09a24b16290f6be95

Joke: I’d tell you a caching joke, but I already stored it for later.
Rating: 7/10 — solid nerdy pun. It’s quick, clever, and “stored it for later” lands nicely.


## Cell 9: `trace()` and concurrency

Everything so far has been sequential. Concurrency is where the `contextvar` design pays off.

The SDK documentation is explicit on this point: the current trace is tracked via a Python `contextvar`,
which means it works with concurrency automatically. You do not pass the trace into each task, and you do not
coordinate anything by hand.

This cell records a start time, opens a single `trace()` block, and fires three `Runner.run()` calls
concurrently through `asyncio.gather()`. Then it prints all three jokes and the total elapsed time.

This is the same `asyncio.gather()` shape you built in Lecture 5.7. There, it produced separate traces, one
per run. Here it produces one trace with three sibling branches whose spans overlap in time. Open the
dashboard afterwards and look at the timeline: the overlap is the visual proof that these ran in parallel.

In [9]:
start = time.time()

with trace("Parallel joke workflow"):
    results = await asyncio.gather(
        Runner.run(agent, "Tell me a joke about Python."),
        Runner.run(agent, "Tell me a joke about SQL."),
        Runner.run(agent, "Tell me a joke about Docker."),
    )

elapsed = time.time() - start

for i, r in enumerate(results, 1):
    print(f"Joke {i}: {r.final_output}")

print(f"\nAll three ran inside ONE trace in {elapsed:.1f}s")

Joke 1: Why did the Python developer wear glasses?  
Because they couldn’t C.
Joke 2: Why did SQL break up with NoSQL?  

It couldn’t handle the lack of relationships.
Joke 3: Why did Docker break up with the server?

It needed more space.

All three ran inside ONE trace in 1.0s


## Cell 10: `custom_span()` for work that is not an agent run

A real workflow is rarely only agent calls. There is a database lookup, a file to parse, an external API to
hit. Those steps take time, and by default that time is invisible in your trace: you see a gap between agent
spans with nothing to explain it.

`custom_span()` fills the gap. Verified signature:

```python
custom_span(
    name: str,
    data: dict[str, Any] | None = None,
    span_id: str | None = None,
    parent: Trace | Span[Any] | None = None,
    disabled: bool = False,
) -> Span[CustomSpanData]
```

The `data` parameter is documented as arbitrary structured data to associate with the span, so you can attach
whatever context makes the span useful later.

This cell opens a `trace()` block. Inside it, it opens a `custom_span` named `database_lookup` carrying a
small `data` dictionary, sleeps briefly to simulate real work, then runs the agent normally after the span
closes.

Note what you did **not** have to do: you never told the span which trace it belongs to. Spans automatically
become part of the current trace and nest under the nearest current span, tracked through the same
`contextvar`. The `parent` argument exists for the rare case where you need to override that.

Worth keeping in proportion: the SDK docs note that in general you do not need to create spans manually. The
SDK already emits agent, generation, function, handoff, guardrail, task and turn spans for you.
`custom_span()` is for the work the SDK cannot see.

In [10]:
with trace("Workflow with custom span"):
    with custom_span(
        "database_lookup",
        data={"table": "jokes", "rows": 42},
    ):
        await asyncio.sleep(0.5)
        print("Simulated database lookup complete")

    result = await Runner.run(agent, "Tell me a joke about indexes.")
    print("Joke:", result.final_output)

Simulated database lookup complete
Joke: Why did the database index break up with the table?

It found the relationship too clustered.


## Cell 11: The gotcha — `custom_span()` outside a trace is a silent no-op

This is the cell to slow down for.

`custom_span()` depends on a trace being active. Call it with no active trace and it does not raise, it does
not warn on your output, and it does not record. Verified from `src/agents/tracing/provider.py`, in
`create_span`:

```python
if not parent:
    current_span = Scope.get_current_span()
    current_trace = Scope.get_current_trace()
    if current_trace is None:
        _safe_debug(
            "No active trace. Make sure to start a trace with "
            "`trace()` first Returning NoOpSpan."
        )
        return NoOpSpan(span_data)
```

You get a `NoOpSpan` back. It satisfies the same interface, so your code keeps running exactly as before. It
simply records nothing. And the one hint the SDK gives you goes through `logger.debug`, so unless you have
turned on debug logging you will never see it.

This cell creates a span twice and prints the class name each time. Once with no trace active, and once
inside a `trace()` block. Compare the two type names.

Notice that the second span is opened with `with custom_span(...) as span_inside:`, not a bare assignment.
That is not a style choice, it is a second gotcha sitting right next to the first one. `custom_span()` never
starts itself. Its own docstring says so directly: "the span will not be started automatically, you should
either do `with custom_span() ...` or call `span.start()` + `span.finish()` manually." A span only gets
queued for export once it starts, through `on_span_start` and `on_span_end`, and neither of those fires until
`.start()` and `.finish()` run, whether you call them yourself or let `with` do it for you. Get back a real
`SpanImpl` instead of a `NoOpSpan` and it is easy to assume the hard part is over. It is not: an unstarted
span is created but never recorded, which looks identical to success right up until you check the dashboard
and it is empty.

One reassurance so you do not over-correct: inside a `Runner.run()` there is always a trace, because the SDK
creates one when none exists. So a `custom_span()` inside a tool function is fine. It is standalone calls
outside any run and outside any `trace()` block that silently vanish.

In [11]:
# No active trace here
span = custom_span("orphan_span", data={"note": "no trace"})
print(f"Span type outside a trace: {type(span).__name__}")
print("This is a NoOpSpan. It records nothing and raises no error.")

with trace("Span parent demo"):
    with custom_span("real_span", data={"note": "has trace"}) as span_inside:
        print(f"\nSpan type inside a trace: {type(span_inside).__name__}")

Span type outside a trace: NoOpSpan
This is a NoOpSpan. It records nothing and raises no error.

Span type inside a trace: SpanImpl


## Cell 12: `flush_traces()` for immediate export

Trace data is not sent the instant a span closes. The default `BatchTraceProcessor` exports traces in the
background every few seconds, or sooner when the in-memory queue reaches its size trigger, and it also
performs a final flush when the process exits.

In a notebook that is invisible. In a long-running worker such as Celery, RQ, Dramatiq, or a FastAPI
background task, it means traces usually do arrive, but not necessarily right after each job finishes. When a
job needs an immediate delivery guarantee, `flush_traces()` gives you one. It blocks until currently
buffered traces and spans have been exported.

This cell runs the agent inside a `trace()` block, then calls `flush_traces()` **after** the block closes.

Look at where that call sits. It is outside the `with` block on purpose. The docs are explicit: call
`flush_traces()` after `trace()` closes, otherwise you flush a partially built trace. You can skip the call
entirely when the default export latency is acceptable, which in a notebook it usually is.

In [12]:
with trace("Flush demo"):
    result = await Runner.run(
        agent,
        "Tell me a joke about garbage collection.",
    )
    print("Joke:", result.final_output)

flush_traces()
print("Traces flushed. Export completed synchronously.")

Joke: Garbage collection walked into a bar.

It said, “I’ll clean up later.”
Traces flushed. Export completed synchronously.


---

## Reference: starting and finishing a trace manually

A trace has to be started and finished. There are two ways to do it.

The first is what you have used all notebook: a context manager, `with trace(...)`. It starts the trace on
entry and finishes it on exit, including when an exception is raised. This is the recommended approach.

The second is manual. `trace.start()` and `trace.finish()` exist and can be called directly. If you go that
route you must pass `mark_as_current=True` to `start()` and `reset_current=True` to `finish()`, otherwise the
current-trace `contextvar` is never updated and nothing nests. That is exactly the bookkeeping the context
manager does for you.

There is no code cell for this. The manual form is documented for completeness, and the docs recommend the
context manager. Use `with trace(...)`.

---

## Reference: `trace()` versus `RunConfig`

Both tools shape what appears in the dashboard. They answer different questions.

| Scenario | Use |
|---|---|
| One `Runner.run()`, and you want a meaningful name | `RunConfig(workflow_name=...)` |
| Multiple runs that form one logical workflow | `with trace("...")` |
| Multiple runs kept as separate traces but loosely linked | `RunConfig(group_id=...)` |
| Non-agent work timed inside the trace tree | `custom_span()` inside an active trace |
| Immediate export guarantee in a worker | `flush_traces()` after the trace context closes |
| One deep link to a whole workflow | `gen_trace_id()` plus `trace(..., trace_id=...)` |

The shortest version of the rule: if the unit you care about is one run, configure the run. If the unit you
care about is the workflow, open a trace.

### One last note on the other span constructors

`custom_span()` is not the only span function in the SDK. There is also `agent_span`, `generation_span`,
`function_span`, `handoff_span`, `guardrail_span`, `task_span`, `turn_span`, `response_span` and
`mcp_tools_span`. You will almost never call them. The SDK creates them for you as your agents run, which is
why the trace trees in Lecture 6.2 were already detailed before you wrote a single tracing line.